# Divide a big file into chunks and rename columns

In [0]:
source_path = (
    "abfss://raw@storageaccountspotifyuk.dfs.core.windows.net/"
    "full/Hospital_General_Information.csv"
)
df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(source_path)

In [0]:
df.printSchema()

In [0]:
column_maps = {i: i.split("/")[0].replace(" ","_").lower() for i in df.columns}
column_maps

In [0]:
df = df.withColumnsRenamed(column_maps)
df.columns

In [0]:
df.rdd.getNumPartitions()  # Not supported in serverless mode, consider using df.repartition(n).count() or other DataFrame methods

In [0]:
# by default, databricks uses 2 partitions
target_path = (
    "abfss://raw@storageaccountspotifyuk.dfs.core.windows.net/"
    "chunk_csv"
)
df.write.format("csv").option("header", True).mode("overwrite").save(target_path)


In [0]:
# 2 partitions to 1
df.coalesce(1) \
  .write \
  .option("header", True) \
  .mode("overwrite") \
  .csv(target_path)

In [0]:
df.repartition(10).write\
    .option("header", True)\
    .mode("overwrite")\
    .csv(target_path)